# 🧠 Week 7 Lab — Student Version## Naive Bayes: When Assumptions Meet Motor Control DataLast week we added 80 motor cortex neurons and discovered a double dissociation: neural features excel at direction decoding (95.8% population vector), while EMG captures impairment better (85.0% LR, AUC 0.94). But logistic regression draws a decision boundary without modeling what healthy or impaired movement *looks like* — and the population vector gives no measure of confidence.This week we introduce **Naive Bayes (NB)**, our first **generative classifier**. NB builds explicit profiles of each class and uses Bayes' theorem to compute posterior probabilities. We'll see NB match or beat LR on three tasks — then collapse catastrophically on the fourth. Understanding *why* it fails will connect motor synergies (Week 4) to machine learning assumptions in a way that changes how you think about feature selection in motor control research.**Data:** `week6_data.pkl` (same dataset as Week 6 — no new file needed)

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from scipy.stats import norm

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, cross_val_predict, LeaveOneGroupOut
from sklearn.metrics import accuracy_score, roc_curve, auc

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Load the dataset
with open('week6_data.pkl', 'rb') as f:
    data = pickle.load(f)

# Extract all arrays
X_raw = data['X_raw']           # EMG features (480, 6)
neural_rates = data['neural_rates']  # Neural features (480, 80)
targets = data['targets']       # Direction labels (480,)
labels = data['labels']         # 'healthy' or 'impaired' (480,)
subjects = data['subjects']     # Subject IDs (480,)
muscle_names = data['muscle_names']
target_angles = data['target_angles']      # radians for 8 directions
dir_degrees = np.array([int(np.degrees(a)) for a in target_angles])  # [0, 45, ..., 315]  # 6 muscle names
neuron_pds = data['neuron_pds']      # Preferred directions for 80 neurons

# Binary labels for clinical task
group_binary = (labels == 'impaired').astype(int)

# Cross-validation setup
logo = LeaveOneGroupOut()

print(f"EMG features: {X_raw.shape}")
print(f"Neural features: {neural_rates.shape}")
print(f"Subjects: {len(np.unique(subjects))} unique")
print(f"Directions: {dir_degrees} (stored as indices 0-7)")
print(f"Groups: {np.unique(labels)} ({group_binary.sum()} impaired, {(1-group_binary).sum()} healthy)")

---

## 🟢 Part 1: Bayes' Theorem in Action (Lecture §1–2)The lecture opened with a clinical scenario: a patient's deltoid shows elevated activation after a stroke. Your instinct says "impaired" — but Bayes' theorem says otherwise. In this part you'll work through the base rate fallacy by hand, then apply the same logic to decode neural activity from 5 neurons.

### Exercise 1.1: The deltoid screening problem — counting patients**Learning objective:** Experience the base rate fallacy firsthand by counting patients, before seeing any equations.A rehabilitation clinic sees 100 referrals. 20% have genuine motor impairment. Among impaired patients, 80% show elevated deltoid activation. Among healthy patients, 30% also show elevated deltoid activation (they just reached hard).**Task:** Compute P(impaired | elevated deltoid activation) by counting. Fill in the numbers below.

In [ ]:
# Exercise 1.1: Count patients to compute the posterior

n_total = 100
n_impaired = ### YOUR CODE HERE ###   # How many of the 100 are truly impaired?
n_healthy = ### YOUR CODE HERE ###     # How many are healthy?

# How many in each group show elevated deltoid activation?
n_impaired_elevated = ### YOUR CODE HERE ###   # 80% of impaired
n_healthy_elevated = ### YOUR CODE HERE ###    # 30% of healthy

# Total patients with elevated deltoid
n_elevated_total = n_impaired_elevated + n_healthy_elevated

# Posterior: of those with elevated deltoid, how many are truly impaired?
posterior = n_impaired_elevated / n_elevated_total

print(f"Impaired with elevated deltoid: {n_impaired_elevated}")
print(f"Healthy with elevated deltoid:  {n_healthy_elevated}")
print(f"Total with elevated deltoid:    {n_elevated_total}")
print(f"\nP(impaired | elevated deltoid) = {n_impaired_elevated}/{n_elevated_total} = {posterior:.1%}")
print(f"\nThe patient is more likely HEALTHY than impaired, despite the elevated deltoid!")

### Exercise 1.2: Verify with Bayes' equation**Learning objective:** Connect the counting approach from 1.1 to the formal Bayes' equation and identify each term (prior, likelihood, evidence, posterior).Now compute the same answer using the formula from the lecture:**P(impaired | elevated) = P(elevated | impaired) × P(impaired) / P(elevated)**where P(elevated) = P(elevated | impaired) × P(impaired) + P(elevated | healthy) × P(healthy)

In [ ]:
# Exercise 1.2: Bayes' theorem calculation

prior_impaired = ### YOUR CODE HERE ###           # P(impaired) = prevalence
likelihood = ### YOUR CODE HERE ###               # P(elevated | impaired) = sensitivity
p_elevated_healthy = ### YOUR CODE HERE ###       # P(elevated | healthy) = false alarm rate
prior_healthy = ### YOUR CODE HERE ###            # P(healthy) = 1 - prevalence

# Evidence: total probability of elevated deltoid
evidence = likelihood * prior_impaired + p_elevated_healthy * prior_healthy

# Posterior
posterior_bayes = (likelihood * prior_impaired) / evidence

print(f"Prior P(impaired) = {prior_impaired}")
print(f"Likelihood P(elevated|impaired) = {likelihood}")
print(f"P(elevated|healthy) = {p_elevated_healthy}")
print(f"Evidence P(elevated) = {evidence}")
print(f"\nPosterior P(impaired|elevated) = {likelihood} × {prior_impaired} / {evidence} = {posterior_bayes:.1%}")
print(f"\nMatches the counting approach: {posterior_bayes:.1%}")

### Exercise 1.3: Worked example — 5 neurons, 1 trial (Lecture §2)**Learning objective:** Apply Bayes' theorem to actual neural data — computing log-likelihoods, exponentiating, and normalizing to get posterior probabilities. This is exactly what GaussianNB does internally.The lecture walked through NB decoding one trial (Trial 8, a 90° reach) using 5 neurons. Let's reproduce this calculation step by step.

In [ ]:
# Exercise 1.3a: Extract means and variances for 5 neurons under 3 directions
# Use only healthy subjects for the training data

healthy_mask = labels == 'healthy'
healthy_neural = neural_rates[healthy_mask]
healthy_targets = targets[healthy_mask]

# Pick 5 neurons with spread-out preferred directions
neuron_ids = [10, 20, 30, 40, 50]  # indices (0-based); neurons 11, 21, 31, 41, 51 in lecture
directions_to_test = [0, 1, 2]  # indices for 0°, 45°, 90°

print("Neuron | PD     | Mean firing rate by direction")
print("-" * 55)
for nid in neuron_ids:
    pd_deg = neuron_pds[nid]
    rates_by_dir = {}
    for d in directions_to_test:
        mask = healthy_targets == d
        rates_by_dir[d] = healthy_neural[mask, nid].mean()
    print(f"  {nid+1:3d}  | {pd_deg:5.0f}° | " + 
          "  ".join(f"{dir_degrees[d]}°: {rates_by_dir[d]:.1f}" for d in directions_to_test))

In [ ]:
# Exercise 1.3b: Compute log-likelihoods for Trial 8 (a 90° reach)
# The observed firing rates for this trial:
trial_idx = 8
observed_rates = neural_rates[trial_idx, neuron_ids]
true_direction = targets[trial_idx]
print(f"Trial {trial_idx}: true direction = {true_direction}°")
print(f"Observed rates: {observed_rates}")

# For each candidate direction, compute the log-likelihood
# log P(rates | direction) = sum of log P(r_j | direction) for each neuron j
# where P(r_j | direction) is a Gaussian with mean and variance learned from training data

for d in directions_to_test:
    mask = (healthy_targets == d)
    log_lik = 0
    print(f"\nDirection {dir_degrees[d]}°:")
    for j, nid in enumerate(neuron_ids):
        mu = healthy_neural[mask, nid].mean()
        sigma2 = healthy_neural[mask, nid].var()
        # Log-likelihood of observed rate under this Gaussian
        ll_j = ### YOUR CODE HERE ###  # Use: -0.5 * np.log(2*np.pi*sigma2) - 0.5 * (observed_rates[j] - mu)**2 / sigma2
        log_lik += ll_j
        print(f"  Neuron {nid+1} (PD={np.degrees(neuron_pds[nid]):.0f}°): rate={observed_rates[j]:.1f}, "
              f"μ={mu:.1f}, σ²={sigma2:.1f}, log P = {ll_j:.2f}")
    print(f"  Total log-likelihood = {log_lik:.2f}")

In [ ]:
# Exercise 1.3c: Convert log-likelihoods to posteriors
# Step 1: Compute log-likelihoods for all 3 directions (from above)
# Step 2: Subtract the maximum (for numerical stability)
# Step 3: Exponentiate
# Step 4: Normalize to sum to 1

log_liks = []
for d in directions_to_test:
    mask = (healthy_targets == d)
    ll = 0
    for j, nid in enumerate(neuron_ids):
        mu = healthy_neural[mask, nid].mean()
        sigma2 = healthy_neural[mask, nid].var()
        ll += -0.5 * np.log(2 * np.pi * sigma2) - 0.5 * (observed_rates[j] - mu)**2 / sigma2
    log_liks.append(ll)

log_liks = np.array(log_liks)
print("Log-likelihoods:", [f"{ll:.2f}" for ll in log_liks])

# Subtract max for numerical stability, then exponentiate
shifted = ### YOUR CODE HERE ###   # log_liks - log_liks.max()
unnormalized = ### YOUR CODE HERE ###   # np.exp(shifted)
posteriors = ### YOUR CODE HERE ###   # unnormalized / unnormalized.sum()

print("\nPosteriors:")
for d, p in zip(directions_to_test, posteriors):
    print(f"  P(direction={dir_degrees[d]}° | rates) = {p:.3f}")
print(f"\nTrue direction: {dir_degrees[true_direction]}°")
print(f"NB prediction: {dir_degrees[directions_to_test[np.argmax(posteriors)]]}° (correct!)")

### Exercise 1.4: Visualize the posteriors (Lecture Figure 2)**Learning objective:** See the posterior distribution as a bar chart — the full output of NB, richer than a single label. The correct direction should dominate, with uncertainty reflected in neighboring directions.Plot the posterior distribution as a bar chart.

In [ ]:
# Exercise 1.4: Posterior bar chart
# Reproduce the worked example panel from Lecture Figure 2

fig, ax = plt.subplots(figsize=(6, 4))

### YOUR CODE HERE ###
# - Bar chart with directions_to_test on x-axis, posteriors on y-axis
# - Color the winning direction differently
# - Add percentage labels on each bar
# - Title: "NB Posterior: Trial 8 (true direction = XX°)"

plt.tight_layout()
plt.show()

---

## 🟢 Part 2: How NB Learns — Profiles and Assumptions (Lecture §2–3)In Part 1, we computed NB by hand for 5 neurons. Now let's see what sklearn's GaussianNB actually learns when trained on the full dataset — and critically, check whether its independence assumption holds for EMG vs neural features.

### Exercise 2.1: Inspect GaussianNB's learned parameters**Learning objective:** Demystify GaussianNB by seeing what it stores — just means and variances for each feature under each class. This is the "lookup table" the lecture described (640 means + 640 variances for 80 neurons × 8 directions).

In [ ]:
# Exercise 2.1: Train and inspect GaussianNB

scaler = StandardScaler()
X_neural_scaled = scaler.fit_transform(neural_rates)

nb = GaussianNB()
nb.fit(X_neural_scaled, targets)

print(f"Classes: {nb.classes_}")
print(f"Class priors: {nb.class_prior_}")
print(f"Means shape (theta_): {nb.theta_.shape}  — {nb.theta_.shape[0]} directions × {nb.theta_.shape[1]} neurons")
print(f"Variances shape (var_): {nb.var_.shape}")
print(f"\nTotal parameters: {nb.theta_.size + nb.var_.size} "
      f"({nb.theta_.size} means + {nb.var_.size} variances)")

# Show means for neuron 21 (PD ≈ 90°) across all directions
neuron_idx = 20  # 0-based
print(f"\nNeuron 21 (PD ≈ {np.degrees(neuron_pds[neuron_idx]):.0f}°) — mean firing rate by direction:")
for i, d in enumerate(nb.classes_):
    print(f"  {d:>3}°: μ = {nb.theta_[i, neuron_idx]:+.2f} (scaled)")

### Exercise 2.2: Visualize class-conditional Gaussians (Lecture Figure 6)**Learning objective:** See that each neuron has a different Gaussian for each direction, and that neurons with different preferred directions provide *different evidence* about the reach. This is why NB works — each feature votes independently.Plot the Gaussians for 3 neurons with different preferred directions.

In [ ]:
# Exercise 2.2: Class-conditional Gaussians for 3 neurons
# Reproduce Lecture Figure 6

neuron_indices = [10, 30, 60]  # Neurons 11 (PD≈45°), 31 (PD≈135°), 61 (PD≈270°)
directions_to_show = [0, 2, 4]  # indices for 0°, 90°, 180°  # 3 directions for clarity
colors = {'0': '#e74c3c', '90': '#2ecc71', '180': '#3498db'}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, nid in zip(axes, neuron_indices):
    for d in directions_to_show:
        mask = targets == d
        rates = neural_rates[mask, nid]
        mu, sigma = rates.mean(), rates.std()

        ### YOUR CODE HERE ###
        # - Create x range: np.linspace(mu - 4*sigma, mu + 4*sigma, 200) (or a fixed range)
        # - Compute Gaussian PDF: norm.pdf(x, mu, sigma)
        # - Plot with label showing direction and mean
        # - Use different colors for each direction

    ax.set_xlabel('Firing Rate (spk/s)')
    ax.set_ylabel('P(rate | direction)')
    ax.set_title(f'Neuron {nid+1} (PD ≈ {np.degrees(neuron_pds[nid]):.0f}°)')
    ax.legend(fontsize=8)

plt.suptitle('Gaussian NB: Each Neuron Has a Different Distribution for Each Direction',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

### Exercise 2.3: Feature correlations — EMG vs Neural (Lecture Figure 5)**Learning objective:** Visualize the key difference between our two feature sets. EMG muscles are highly correlated (r up to 0.99) because of synergies; neurons are more independent. This sets up Part 4's explanation of why NB fails on EMG.

In [ ]:
# Exercise 2.3: Correlation matrices
# Reproduce Lecture Figure 5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) EMG: 6×6 correlation matrix
emg_corr = ### YOUR CODE HERE ###  # np.corrcoef(X_raw.T)

im0 = axes[0].imshow(emg_corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='equal')
# Add correlation values as text
for i in range(6):
    for j in range(6):
        axes[0].text(j, i, f'{emg_corr[i,j]:.2f}', ha='center', va='center',
                     fontsize=7, color='white' if abs(emg_corr[i,j]) > 0.6 else 'black')
axes[0].set_xticks(range(6))
axes[0].set_xticklabels(muscle_names, rotation=45, ha='right', fontsize=8)
axes[0].set_yticks(range(6))
axes[0].set_yticklabels(muscle_names, fontsize=8)
off_diag = emg_corr[np.triu_indices(6, 1)]
axes[0].set_title(f'(a) EMG: 6 Muscles\nMax |r| = {np.abs(off_diag).max():.2f}', fontweight='bold')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# (b) Neural: 80×80 correlation matrix
neural_corr = ### YOUR CODE HERE ###  # np.corrcoef(neural_rates.T)

im1 = axes[1].imshow(neural_corr, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='equal')
np.fill_diagonal(neural_corr, 0)  # zero diagonal for mean calculation
axes[1].set_title(f'(b) Neural: 80 Neurons\nMean |r| = {np.abs(neural_corr).mean():.2f}', fontweight='bold')
axes[1].set_xlabel('Neuron')
axes[1].set_ylabel('Neuron')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.suptitle('Feature Correlations: Why the Independence Assumption Matters', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nEMG: Max off-diagonal |r| = {np.abs(emg_corr[np.triu_indices(6, 1)]).max():.2f}")
print(f"    → Muscles are highly correlated (synergies from Week 4)")
print(f"Neural: Mean |r| = {np.abs(neural_corr).mean():.2f}")
print(f"    → Neurons are more independent (diverse PDs)")

---

## 🟡 Part 3: Testing NB on All Four Tasks (Lecture §3)Part 2 showed NB working on one trial with 5 neurons. Now we scale up: all 480 trials, all features, all four task/feature combinations. The results reveal three wins and one catastrophe.

### Exercise 3.1: Run LOSO for all four tasks (Lecture Figure 3)**Learning objective:** Run the full experiment and reproduce the lecture's central result — NB matches or beats LR on three tasks but collapses to 65.4% on EMG binary diagnosis.

In [ ]:
# Exercise 3.1: LOSO cross-validation for all 4 tasks
# Build pipelines
pipe_nb = Pipeline([('scaler', StandardScaler()), ('nb', GaussianNB())])
pipe_lr = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(C=10, max_iter=1000))])

# Direction decoding
lr_emg_dir = cross_val_score(pipe_lr, X_raw, targets, cv=logo, groups=subjects)
lr_neur_dir = cross_val_score(pipe_lr, neural_rates, targets, cv=logo, groups=subjects)
nb_emg_dir = ### YOUR CODE HERE ###   # cross_val_score for NB on EMG direction
nb_neur_dir = ### YOUR CODE HERE ###  # cross_val_score for NB on neural direction

# Binary diagnosis
lr_emg_bin = cross_val_score(pipe_lr, X_raw, group_binary, cv=logo, groups=subjects)
lr_neur_bin = cross_val_score(pipe_lr, neural_rates, group_binary, cv=logo, groups=subjects)
nb_emg_bin = ### YOUR CODE HERE ###   # cross_val_score for NB on EMG binary
nb_neur_bin = ### YOUR CODE HERE ###  # cross_val_score for NB on neural binary

print("Direction Decoding:")
print(f"  LR  EMG:    {lr_emg_dir.mean():.1%}")
print(f"  LR  Neural: {lr_neur_dir.mean():.1%}")
print(f"  NB  EMG:    {nb_emg_dir.mean():.1%}")
print(f"  NB  Neural: {nb_neur_dir.mean():.1%}")

print("\nBinary Diagnosis:")
print(f"  LR  EMG:    {lr_emg_bin.mean():.1%}")
print(f"  LR  Neural: {lr_neur_bin.mean():.1%}")
print(f"  NB  EMG:    {nb_emg_bin.mean():.1%}  ← The catastrophe")
print(f"  NB  Neural: {nb_neur_bin.mean():.1%}")

In [ ]:
# Exercise 3.1b: Create the results bar chart (Lecture Figure 3)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (a) Direction decoding
methods_dir = ['LR\n(EMG)', 'LR\n(Neural)', 'NB\n(EMG)', 'NB\n(Neural)']
dir_vals = [lr_emg_dir.mean()*100, lr_neur_dir.mean()*100,
            nb_emg_dir.mean()*100, nb_neur_dir.mean()*100]

### YOUR CODE HERE ###
# - Bar chart for direction decoding
# - Add value labels on each bar
# - Add chance level line at 12.5%
# - Y-axis: 'LOSO Accuracy (%)'

# (b) Binary diagnosis
methods_bin = ['LR\n(EMG)', 'LR\n(Neural)', 'NB\n(EMG)', 'NB\n(Neural)']
bin_vals = [lr_emg_bin.mean()*100, lr_neur_bin.mean()*100,
            nb_emg_bin.mean()*100, nb_neur_bin.mean()*100]

### YOUR CODE HERE ###
# - Bar chart for binary diagnosis
# - Highlight the NB EMG bar in red (the catastrophe)
# - Add chance level line at 50%

plt.suptitle('Week 7 Results: Naive Bayes vs Logistic Regression', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 3.2: ROC curves for the binary task (Lecture Figure 4)**Learning objective:** Go beyond accuracy to see how well the posterior probabilities separate healthy from impaired across all thresholds. The ROC curve reveals that NB's posteriors on EMG are poorly calibrated (AUC 0.68) compared to LR (AUC 0.94).

In [ ]:
# Exercise 3.2: ROC curves for binary task (Lecture Figure 4)

# Get predicted probabilities via cross_val_predict
yp_nb_emg = cross_val_predict(pipe_nb, X_raw, group_binary, cv=logo, 
                               groups=subjects, method='predict_proba')[:, 1]
yp_nb_neur = cross_val_predict(pipe_nb, neural_rates, group_binary, cv=logo,
                                groups=subjects, method='predict_proba')[:, 1]
yp_lr_emg = cross_val_predict(pipe_lr, X_raw, group_binary, cv=logo,
                               groups=subjects, method='predict_proba')[:, 1]
yp_lr_neur = cross_val_predict(pipe_lr, neural_rates, group_binary, cv=logo,
                                groups=subjects, method='predict_proba')[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, yp_nb, yp_lr, title in [
    (axes[0], yp_nb_emg, yp_lr_emg, '(a) EMG Features'),
    (axes[1], yp_nb_neur, yp_lr_neur, '(b) Neural Features')]:
    
    ### YOUR CODE HERE ###
    # For each model (NB and LR):
    #   - fpr, tpr, _ = roc_curve(group_binary, yp)
    #   - auc_val = auc(fpr, tpr)
    #   - ax.plot(fpr, tpr, label=f'Model (AUC = {auc_val:.2f})')
    # - Add diagonal reference line
    # - Labels, title, legend
    
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(title, fontweight='bold')
    ax.legend()

plt.suptitle('ROC Curves: How Well Do Posterior Probabilities Separate Groups?',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 3.3: Where does NB get it wrong? (Lecture Figure 7)**Learning objective:** See that NB doesn't make scattered random errors — it systematically misclassifies entire subject clusters. This is the signature of a model whose assumptions are wrong.Plot correct vs misclassified trials in PCA space for both LR and NB.

In [ ]:
# Exercise 3.3: Misclassification plot (Lecture Figure 7)
# Run LOSO and collect predictions for both models

nb_pred_loso = np.zeros(len(group_binary), dtype=int)
lr_pred_loso = np.zeros(len(group_binary), dtype=int)

for subj in np.unique(subjects):
    test = subjects == subj
    train = ~test
    sc = StandardScaler()
    Xtr = sc.fit_transform(X_raw[train])
    Xte = sc.transform(X_raw[test])
    
    nb_tmp = GaussianNB()
    nb_tmp.fit(Xtr, group_binary[train])
    nb_pred_loso[test] = nb_tmp.predict(Xte)
    
    lr_tmp = LogisticRegression(C=10, max_iter=1000)
    lr_tmp.fit(Xtr, group_binary[train])
    lr_pred_loso[test] = lr_tmp.predict(Xte)

# PCA projection for visualization
sc_all = StandardScaler()
X_pca = PCA(n_components=2).fit_transform(sc_all.fit_transform(X_raw))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, pred, name in [(axes[0], lr_pred_loso, 'Logistic Regression'),
                        (axes[1], nb_pred_loso, 'Naive Bayes')]:
    correct = pred == group_binary
    wrong = ~correct
    fp = (wrong & (group_binary == 0)).sum()
    fn = (wrong & (group_binary == 1)).sum()
    acc = 100 * correct.mean()
    
    ### YOUR CODE HERE ###
    # Plot 4 scatter groups:
    # 1. Correct healthy (light blue circles, small, alpha=0.4)
    # 2. Correct impaired (light red circles, small, alpha=0.4)
    # 3. Misclassified healthy → impaired (blue X markers, large, bold)
    # 4. Misclassified impaired → healthy (red X markers, large, bold)
    # Title: model name + accuracy
    
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.legend(fontsize=7.5, loc='lower right')
    ax.grid(True, alpha=0.15)

plt.suptitle('Where Each Model Gets It Wrong (EMG Binary, LOSO)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

### 🤔 Thought ExerciseThe results show three wins and a catastrophe: NB matches or beats LR on EMG direction (81.2%), neural direction (95.2%), and neural binary (77.7%) — but collapses to 65.4% on EMG binary.**Question:** All four tasks use the same NB algorithm. The only things that change are the features (EMG vs neural) and the task (direction vs binary). What property of the EMG features, combined with the binary task's subtle signal, could explain why *this specific combination* fails while the other three succeed?*Hint: Look at the correlation matrices from Exercise 2.3.*

In [ ]:
# Your reflection:
#
#

### Exercise 3.4: Bonus — Bernoulli NB on binarized spike counts**Learning objective:** See how the choice of NB variant depends on the data representation. Bernoulli NB treats each feature as binary (spike / no spike). This preserves directional tuning (neurons fire above baseline for their preferred direction) but destroys the subtle amplitude differences that distinguish healthy from impaired.The syllabus mentioned comparing Gaussian NB on continuous rates with Bernoulli NB on binarized counts. Let's test this: binarize each neuron's rate (above median → 1, below → 0), then run Bernoulli NB on the same tasks.

In [ ]:
# Exercise 3.4: Bernoulli NB on binarized neural rates
from sklearn.naive_bayes import BernoulliNB
from sklearn.preprocessing import Binarizer

# Pipeline: standardize → binarize at 0 (above/below mean) → Bernoulli NB
pipe_bnb = Pipeline([
    ('scaler', StandardScaler()),
    ('binarizer', Binarizer(threshold=0.0)),  # above mean → 1, below → 0
    ('nb', BernoulliNB())
])

# Run on neural direction and binary tasks
bnb_neur_dir = ### YOUR CODE HERE ###  # cross_val_score for Bernoulli NB on neural direction
bnb_neur_bin = ### YOUR CODE HERE ###  # cross_val_score for Bernoulli NB on neural binary

print("Neural Direction Decoding:")
print(f"  Gaussian NB (continuous rates): {nb_neur_dir.mean():.1%}")
print(f"  Bernoulli NB (binarized rates): {bnb_neur_dir.mean():.1%}")
print(f"  Difference: {(nb_neur_dir.mean() - bnb_neur_dir.mean())*100:+.1f} pp")

print("\nNeural Binary Diagnosis:")
print(f"  Gaussian NB (continuous rates): {nb_neur_bin.mean():.1%}")
print(f"  Bernoulli NB (binarized rates): {bnb_neur_bin.mean():.1%}")
print(f"  Difference: {(nb_neur_bin.mean() - bnb_neur_bin.mean())*100:+.1f} pp")

print("\nKey insight: Binarizing preserves DIRECTION information (above/below")
print("baseline maps to preferred/anti-preferred) but destroys the subtle")
print("AMPLITUDE differences between healthy and impaired subjects.")
print("The choice of NB variant must match what your data representation preserves.")

---

## 🟡 Part 4: Why NB Failed — Synergies and Independence (Lecture §4)The correlation matrices from Part 2 showed that EMG muscles are highly correlated while neurons are more independent. Now we connect this directly to NB's failure: correlated features cause double-counting, which inflates posteriors and breaks classification on subtle signals.

### Exercise 4.1: Identify the correlated muscle pairs**Learning objective:** Connect the correlation matrix to specific muscle pairs and to the synergy modules from Week 4. The highest correlations are between single-joint and bi-articular muscles of the same type.

In [ ]:
# Exercise 4.1: Find the highest-correlated muscle pairs
emg_corr = np.corrcoef(X_raw.T)

print("EMG Correlation Matrix — Highest off-diagonal pairs:")
print("-" * 55)
for i in range(6):
    for j in range(i+1, 6):
        if abs(emg_corr[i,j]) > 0.5:
            print(f"  {muscle_names[i]:>10} × {muscle_names[j]:<10}: r = {emg_corr[i,j]:+.2f}"
                  + ("  ← SYNERGY MODULE" if abs(emg_corr[i,j]) > 0.9 else ""))

print(f"\nThe highest correlations are between single-joint and bi-articular")
print(f"muscles of the same type — these are the synergy modules from Week 4.")
print(f"When NB treats them as independent, it counts the same evidence twice.")

### Exercise 4.2: See double-counting in real muscle data**Learning objective:** Understand *why* correlated features break NB by watching it happen with actual muscles from our dataset.The lecture explained that when NB multiplies likelihoods from correlated features, it counts the same evidence twice. But what does that actually look like? In this exercise, you'll train NB three ways and compare the posterior distributions:- **(a) 1 muscle** — Sh.H.Flex alone. Baseline: how much can one feature tell us?- **(b) 2 correlated muscles** — Sh.H.Flex + Bi.Flex (r = 0.99). These are the synergy pair from Week 4. If NB treats them as independent, it counts the same evidence twice. The posteriors should spread out (NB becomes more confident) but accuracy should barely improve — because no new information was added.- **(c) 2 uncorrelated muscles** — Sh.H.Flex + El.Ext (r = −0.17). These carry genuinely different information. Both posteriors AND accuracy should improve.The key comparison: **(b) vs (c)**. Both add a second muscle, but only (c) adds new information. If NB's posteriors spread in (b) without accuracy improving, that's double-counting in action.

In [ ]:
# Exercise 4.2: Double-counting with real muscles
# Compare 3 NB models: 1 muscle, 2 correlated, 2 uncorrelated

sc = StandardScaler()

# (a) 1 muscle: Sh.H.Flex only
X_1 = sc.fit_transform(X_raw[:, [0]])
nb_1 = GaussianNB()
nb_1.fit(X_1, group_binary)
p_1 = nb_1.predict_proba(X_1)[:, 1]

# (b) 2 correlated muscles: Sh.H.Flex + Bi.Flex (r = 0.99)
X_2corr = sc.fit_transform(X_raw[:, [0, 4]])
nb_2corr = ### YOUR CODE HERE ###   # Train GaussianNB
nb_2corr.fit(X_2corr, group_binary)
p_2corr = nb_2corr.predict_proba(X_2corr)[:, 1]

# (c) 2 uncorrelated muscles: Sh.H.Flex + El.Ext (r = -0.17)
X_2uncorr = sc.fit_transform(X_raw[:, [0, 3]])
nb_2uncorr = ### YOUR CODE HERE ###   # Train GaussianNB
nb_2uncorr.fit(X_2uncorr, group_binary)
p_2uncorr = nb_2uncorr.predict_proba(X_2uncorr)[:, 1]

# Print the key comparison
corr_flex = np.corrcoef(X_raw[:, 0], X_raw[:, 4])[0, 1]
corr_ext = np.corrcoef(X_raw[:, 0], X_raw[:, 3])[0, 1]
print(f"Sh.H.Flex × Bi.Flex correlation:  r = {corr_flex:.2f}")
print(f"Sh.H.Flex × El.Ext correlation:   r = {corr_ext:.2f}")

for name, probs in [("1 muscle", p_1), ("2 correlated", p_2corr), ("2 uncorrelated", p_2uncorr)]:
    acc = np.mean((probs > 0.5) == group_binary) * 100
    spread = np.std(probs)
    print(f"\n{name}: accuracy = {acc:.0f}%, posterior spread (std) = {spread:.3f}")

In [ ]:
# Exercise 4.2b: Visualize the three posterior distributions

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

panels = [
    (axes[0], p_1, f'(a) 1 Muscle\n(Sh.H.Flex)'),
    (axes[1], p_2corr, f'(b) 2 Correlated Muscles\n(Sh.H.Flex + Bi.Flex, r={corr_flex:.2f})'),
    (axes[2], p_2uncorr, f'(c) 2 Uncorrelated Muscles\n(Sh.H.Flex + El.Ext, r={corr_ext:.2f})')
]

for ax, probs, title in panels:
    ### YOUR CODE HERE ###
    # - Histogram of probs for healthy (blue, alpha=0.6) and impaired (red, alpha=0.6)
    #   Use density=True, bins=25, range=(0,1)
    # - Vertical dashed line at 0.5 (decision boundary)
    # - Compute and show accuracy in the title
    # - Labels: x = 'P(impaired)', y = 'Density'
    
    ax.legend(fontsize=8)
    ax.set_xlim(-0.05, 1.05)

plt.suptitle('Double-Counting: Correlated Muscle Spreads Posteriors Without Improving Accuracy',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print("\nKey insight: Panel (b) posteriors spread wider than (a) — NB is MORE CONFIDENT.")
print("But accuracy barely changes (56% → 57%). The extra confidence is FALSE.")
print("Panel (c) posteriors spread AND accuracy jumps to 72% — that's REAL new evidence.")

### Exercise 4.3: Discriminative vs Generative boundaries (Lecture Figure 1)**Learning objective:** See the structural difference between discriminative (LR) and generative (NB) classifiers. LR learns a linear boundary; NB fits elliptical Gaussians and the boundary emerges from where the distributions cross.Plot both approaches side by side on our binary task data in PCA space.

In [ ]:
# Exercise 4.3: Decision boundaries in PCA space (Lecture Figure 1)
# Use the first 2 PCA components of EMG data

sc = StandardScaler()
X_s = sc.fit_transform(X_raw)
pca2 = PCA(n_components=2)
X_pca2 = pca2.fit_transform(X_s)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, clf, title in [(axes[0], LogisticRegression(C=10, max_iter=1000), '(a) Logistic Regression'),
                        (axes[1], GaussianNB(), '(b) Gaussian Naive Bayes')]:
    clf.fit(X_pca2, group_binary)
    
    # Create mesh grid for decision boundary
    xlim = (X_pca2[:,0].min()-1, X_pca2[:,0].max()+1)
    ylim = (X_pca2[:,1].min()-1, X_pca2[:,1].max()+1)
    xx, yy = np.meshgrid(np.linspace(*xlim, 200), np.linspace(*ylim, 200))
    Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    
    ### YOUR CODE HERE ###
    # - contourf for confidence shading (cmap='RdBu_r', alpha=0.3)
    # - contour for decision boundary at Z=0.5
    # - scatter healthy (blue) and impaired (red)
    # - labels, title, legend
    
plt.suptitle('Decision Boundaries: Same Data, Different Assumptions', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### 🤔 Thought ExerciseIn Week 4, you extracted motor synergies using NMF and found that muscles within the same synergy module co-activate with correlations up to r = 0.99.**Question:** The lecture states that NB's double-counting problem is a direct consequence of these synergies. In your own words, explain the chain of reasoning: synergies → correlations → double-counting → wrong posteriors → 65.4% accuracy. Why does direction decoding (81.2%) survive this problem while binary diagnosis does not?

In [ ]:
# Your reflection:
#
#

---

## 🔴 Part 5: PCA to the Rescue — Decorrelation Fixes NB (Lecture §5)If NB fails because features are correlated, and PCA produces uncorrelated components by construction, then PCA + NB should fix the problem. In this part you'll test the rescue, visualize decorrelation, and discover that PCA serves completely different purposes depending on the downstream model.

### Exercise 5.1: PCA(6) + NB on EMG binary (Lecture Figure 8)**Learning objective:** See the rescue — PCA with all 6 components (retaining all information, just decorrelating) brings NB from 65% to 85%, matching LR.

In [ ]:
# Exercise 5.1: PCA + NB rescue

# Build PCA + NB pipeline
pipe_pca_nb = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=6)),
    ('nb', GaussianNB())
])

pca_nb_emg_bin = cross_val_score(pipe_pca_nb, X_raw, group_binary, cv=logo, groups=subjects)

print(f"NB Raw EMG Binary:       {nb_emg_bin.mean():.1%}")
print(f"NB PCA(6) EMG Binary:    {pca_nb_emg_bin.mean():.1%}  ← +{(pca_nb_emg_bin.mean()-nb_emg_bin.mean())*100:.0f} pp!")
print(f"LR Raw EMG Binary:       {lr_emg_bin.mean():.1%}")
print(f"\nPCA decorrelation rescues NB to match LR!")

In [ ]:
# Exercise 5.1b: Create Figure 8 — rescue bars + ROC curves

# Get PCA+NB predicted probabilities for ROC
yp_pca_nb_emg = cross_val_predict(pipe_pca_nb, X_raw, group_binary, cv=logo,
                                   groups=subjects, method='predict_proba')[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (a) Bar chart: Raw NB vs PCA+NB vs LR
bar_labels = ['NB\n(Raw EMG)', 'NB\n(PCA+EMG)', 'LR\n(Raw EMG)']
bar_vals = [nb_emg_bin.mean()*100, pca_nb_emg_bin.mean()*100, lr_emg_bin.mean()*100]
bar_colors = ['#e74c3c', '#2ecc71', '#2E8B8B']

### YOUR CODE HERE ###
# - Bar chart with the 3 values
# - Annotate the +20pp gain
# - Add chance line at 50%

# (b) ROC curves: NB raw vs PCA+NB vs LR
### YOUR CODE HERE ###
# - 3 ROC curves with AUC in legend
# - Diagonal reference line

plt.suptitle('The Fix: PCA Removes Correlations, NB Recovers', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 5.2: Visualize decorrelation**Learning objective:** Confirm *why* PCA rescues NB by showing the correlation matrix before and after PCA. The off-diagonal correlations vanish completely.

In [ ]:
# Exercise 5.2: Correlation matrix before vs after PCA

sc = StandardScaler()
X_scaled = sc.fit_transform(X_raw)
pca6 = PCA(n_components=6)
X_pca6 = pca6.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before PCA
corr_before = np.corrcoef(X_scaled.T)
im0 = axes[0].imshow(corr_before, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title('Before PCA (Raw EMG)', fontweight='bold')
axes[0].set_xticks(range(6))
axes[0].set_xticklabels(muscle_names, rotation=45, ha='right', fontsize=8)
axes[0].set_yticks(range(6))
axes[0].set_yticklabels(muscle_names, fontsize=8)
plt.colorbar(im0, ax=axes[0])

# After PCA
corr_after = ### YOUR CODE HERE ###  # np.corrcoef(X_pca6.T)
im1 = axes[1].imshow(corr_after, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title('After PCA (6 Components)', fontweight='bold')
axes[1].set_xticks(range(6))
axes[1].set_xticklabels([f'PC{i+1}' for i in range(6)], fontsize=8)
axes[1].set_yticks(range(6))
axes[1].set_yticklabels([f'PC{i+1}' for i in range(6)], fontsize=8)
plt.colorbar(im1, ax=axes[1])

plt.suptitle('PCA Decorrelation: Off-Diagonal Correlations Vanish', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Max off-diagonal |r| before PCA: {np.abs(corr_before[np.triu_indices(6,1)]).max():.2f}")
print(f"Max off-diagonal |r| after PCA:  {np.abs(corr_after[np.triu_indices(6,1)]).max():.6f}")

### Exercise 5.3: How many PCA components does NB need?**Learning objective:** Discover that NB needs *all 6* PCA components to match LR — because the diagnostic signal lives in later PCs (recall Week 5's PC4).Sweep from 1 to 6 PCA components and plot the accuracy curve.

In [ ]:
# Exercise 5.3: PCA component sweep

n_components_range = range(1, 7)
pca_nb_accs = []

for n_comp in n_components_range:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=n_comp)),
        ('nb', GaussianNB())
    ])
    scores = cross_val_score(pipe, X_raw, group_binary, cv=logo, groups=subjects)
    pca_nb_accs.append(scores.mean() * 100)
    print(f"  PCA({n_comp}) + NB: {scores.mean():.1%}")

### YOUR CODE HERE ###
# Line plot: n_components on x-axis, accuracy on y-axis
# Add horizontal reference lines for:
#   - Raw NB (65.4%)
#   - LR baseline (85.0%)
#   - Chance (50%)
# Title: "How Many PCA Components Does NB Need?"

plt.show()

### Exercise 5.4: PCA's three roles across the course (Lecture Figure 9)**Learning objective:** Synthesize PCA's three roles across Weeks 4, 5, and 7 — understanding, feature input, and decorrelation — with opposite outcomes depending on context.

In [ ]:
# Exercise 5.4: Summarize PCA's three roles
# This is a reflection + printout exercise

print("=" * 65)
print("PCA's Three Roles Across the Course")
print("=" * 65)
print()
print("Week 4: Discover synergy structure")
print("  → 2 PCs revealed 3 healthy vs 2 impaired synergy modules")
print("  → Purpose: understanding, not prediction")
print()
print("Week 5: Feature input to LR")
print("  → 2 PCs DESTROYED the diagnostic signal (AUC 0.92 → 0.36)")
print("  → Information lived in PC4, which was discarded")
print()
print("Week 7: Decorrelation for NB")
print("  → 6 PCs RESCUED NB accuracy (65% → 85%)")
print("  → Same information, just uncorrelated")
print()
print("Lesson: PCA is not inherently good or bad.")
print("Its value depends on what the downstream model needs.")

---

## 🔴 Part 6: Bringing It Together (Lecture §6–7)The final part compiles all results into the running comparison table and asks you to think about the practical implications for motor control research.

### Exercise 6.1: The comparison table (Lecture Figure 10)**Learning objective:** See the full picture — two algorithms × two feature sets, with the pattern now clear: assumptions about data structure matter as much as the algorithm itself.

In [ ]:
# Exercise 6.1: Comparison table

print("=" * 75)
print("Running Comparison Table — Through Week 7")
print("=" * 75)
print(f"{'Week':<6} {'Method':<22} {'Features':<14} {'8-Dir':>8} {'Binary':>8} {'AUC':>6}")
print("-" * 75)

# Week 5 baselines
print(f"{'5':<6} {'Logistic Reg (C=10)':<22} {'Raw EMG (6)':<14} {'80.8%':>8} {'85.0%':>8} {'0.94':>6}")
print(f"{'6':<6} {'Logistic Reg (C=10)':<22} {'Neural (80)':<14} {'91.2%':>8} {'73.3%':>8} {'0.80':>6}")
print(f"{'6':<6} {'Population Vector':<22} {'Neural (80)':<14} {'95.8%':>8} {'N/A':>8} {'N/A':>6}")

# Week 7 NB results
auc_nb_emg = auc(*roc_curve(group_binary, yp_nb_emg)[:2])
auc_nb_neur = auc(*roc_curve(group_binary, yp_nb_neur)[:2])
auc_pca_nb = auc(*roc_curve(group_binary, yp_pca_nb_emg)[:2])

print(f"{'7':<6} {'Gaussian NB':<22} {'Raw EMG (6)':<14} {nb_emg_dir.mean():>7.1%} {nb_emg_bin.mean():>7.1%} {auc_nb_emg:>6.2f}")
print(f"{'7':<6} {'Gaussian NB':<22} {'Neural (80)':<14} {nb_neur_dir.mean():>7.1%} {nb_neur_bin.mean():>7.1%} {auc_nb_neur:>6.2f}")
print(f"{'7':<6} {'PCA(6) + NB':<22} {'Raw EMG (6)':<14} {'—':>8} {pca_nb_emg_bin.mean():>7.1%} {auc_pca_nb:>6.2f}")
print("-" * 75)

### 🤔 Final Thought Exercise: NMF vs PCA for Motor ControlThe lecture (Section 6) argued that for EMG data, NMF might be a better preprocessing step than PCA because NMF produces physiologically interpretable synergy modules, while PCA produces abstract components with negative weights.**Question:** Imagine you are building a clinical screening tool that uses EMG from a reaching task to identify early motor impairment. You plan to use Naive Bayes as your classifier. Would you preprocess with PCA or NMF? Consider:1. **Interpretability**: Can a clinician understand and act on the features?2. **Independence**: Which decomposition better satisfies NB's assumption?3. **Clinical communication**: How would you explain a positive result to the patient?There is no single right answer — the goal is to reason through the tradeoffs.

In [ ]:
# Your reflection:
#
#

---

## SummaryThis lab followed the 7 sections of the Week 7 lecture:1. **§1–2 (Part 1):** Bayes' theorem through the deltoid screening example, then a 5-neuron worked example computing posteriors by hand.2. **§2–3 (Part 2):** Inspected GaussianNB's learned parameters (640 means + 640 variances) and checked the independence assumption via correlation matrices.3. **§3 (Part 3):** Tested NB on all four tasks — three wins and a catastrophe (65.4% on EMG binary).4. **§4 (Part 4):** Traced the failure to synergy-driven correlations and double-counting, simulated the overconfidence effect.5. **§5 (Part 5):** PCA decorrelation rescued NB from 65% to 85%, matching LR.6. **§6–7 (Part 6):** Updated the comparison table and reflected on NMF vs PCA for clinical motor control.**Key takeaway:** Before trusting an algorithm's output, check whether its assumptions match your data. The motor synergies from Week 4 directly caused the ML failure in Week 7 — and understanding that connection makes you a better researcher in both domains.